In [1]:
import os
from datetime import datetime
import pytz
import pandas as pd
from supabase import create_client
from dotenv import  load_dotenv

In [2]:
SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

In [3]:
supabase = create_client(SUPABASE_URL,SUPABASE_KEY)

In [4]:
def current_ist_date():
    ist = pytz.timezone("Asia/Kolkata")
    return datetime.now(ist).strftime("%Y-%m-%d")

In [5]:
print(current_ist_date())

2026-03-06


In [6]:
def create_classroom(class_name,code="1234",daily_limit=10):
    existing = (
        supabase.table("classroom_setting")
        .select("*")
        .eq("class_name",class_name)
        .execute()
        .data
    )

    if existing:
        print(f"Class '{class_name}' already exists.")
        return
    
    supabase.table("classroom_setting").insert({
        "class_name" : class_name,
        "code" : code,
        "daily_limit" : daily_limit,
        "is_open" : True
    }).execute()

    print(f"Classroom '{class_name}' created successfully")


In [7]:
create_classroom("Demo_class2",code="4264",daily_limit=5)

Class 'Demo_class2' already exists.


In [8]:
def add_student(class_name,roll_number,name):
    existing = (
        supabase.table("roll_map")
        .select("*")
        .eq("roll_number",roll_number)
        .execute()
        .data
    )

    if existing:
        print(f"Roll {roll_number} already assigned to {existing[0]['name']}")
        return
    

    supabase.table("roll_map").insert({
        "class_name" : class_name,
        "roll_number": roll_number,
        "name" : name
    }).execute()

    print(f"Added {name} (Roll {roll_number}) to {class_name}")

In [9]:
add_student("Demo_class2","1","saurav")
add_student("Demo_class2","2","lokesh")
add_student("Demo_class2","3","dipesh")

Roll 1 already assigned to saurav
Roll 2 already assigned to lokesh
Roll 3 already assigned to dipesh


In [10]:
add_student("Demo_class2","1","sam")

Roll 1 already assigned to saurav


In [11]:
def mark_attendance(class_name,roll_number,code):
    today = current_ist_date()

    settings = (
        supabase.table("classroom_setting")
        .select("*")
        .eq("class_name",class_name)
        .execute()
        .data[0]
    )

    if code != settings["code"]:
        print("Incorrect code entered.")
        return
    
    existing = (
        supabase.table("attendance")
        .select("*")
        .eq("class_name" , class_name)
        .eq("roll_number",roll_number)
        .eq("date",today)
        .execute()
        .data
    )


    if existing:
        print("Attendance already marked today.")
        return
    

    count_today = (
        supabase.table("attendance")
        .select("*",count="exact")
        .eq("class_name",class_name)
        .eq("date", today)
        .execute()
        .count
    )

    if count_today >= settings["daily_limit"]:
        print("Daily attendance limit reached.")
        return
    

    student = (
        supabase.table("roll_map")
        .select("*")
        .eq("class_name",class_name)
        .eq("roll_number" , roll_number)
        .execute()
        .data
    )

    if not student:
        print("Roll number not registered")
        return
    

    name = student[0]["name"]

    supabase.table("attendance").insert({
        "class_name" : class_name,
        "roll_number": roll_number,
        "name": name,
        "date" : today
    }).execute()


    print(f"Attendance recorded for {name} ({roll_number})")

In [12]:
mark_attendance("Demo_class2","1","4264")
mark_attendance("Demo_class2","2","4264")
mark_attendance("Demo_class2","3","4264")

Attendance recorded for saurav (1)
Attendance recorded for lokesh (2)
Attendance recorded for dipesh (3)


In [16]:
def attendance_matrix(class_name):
    records = (
        supabase.table("attendance")
        .select("*")
        .eq("class_name",class_name)
        .order("date",desc=True)
        .execute()
        .data
    )

    if not records:
        print("No attendance found.")
        return
    
    df = pd.DataFrame(records)
    df["status"] = "P"

    pivot_df = df.pivot_table(
        index=["roll_number","name"],
        columns="date",
        values="status",
        aggfunc="first",
        fill_value="A"
    ).reset_index()

    return pivot_df

In [17]:
attendance_matrix("Demo_class2")

date,roll_number,name,2026-03-05,2026-03-06
0,1,saurav,P,P
1,2,lokesh,P,P
2,3,dipesh,P,P


In [18]:
def attendance_analytics(class_name):
    records = supabase.table("attendance").select("*").eq("class_name", class_name).execute().data
    if not records:
        print("No attendance found")
        return

    df = pd.DataFrame(records)
    df["status"] = "P"
    pivot_df = df.pivot_table(index=["roll_number","name"], columns="date", values="status", aggfunc="first", fill_value="A").reset_index()
    
    date_cols = pivot_df.columns[2:]
    pivot_df["Present_Count"] = pivot_df[date_cols].apply(lambda row: sum(val=="P" for val in row), axis=1)
    pivot_df["Attendance %"] = (pivot_df["Present_Count"]/len(date_cols)*100).round(2)
    
    print("Top 3 students:")
    print(pivot_df.sort_values("Attendance %", ascending=False).head(3))
    
    print("\nBottom 3 students:")
    print(pivot_df.sort_values("Attendance %").head(3))

In [19]:
attendance_analytics("Demo_class2")

Top 3 students:
date roll_number    name 2026-03-05 2026-03-06  Present_Count  Attendance %
0              1  saurav          P          P              2         100.0
1              2  lokesh          P          P              2         100.0
2              3  dipesh          P          P              2         100.0

Bottom 3 students:
date roll_number    name 2026-03-05 2026-03-06  Present_Count  Attendance %
0              1  saurav          P          P              2         100.0
1              2  lokesh          P          P              2         100.0
2              3  dipesh          P          P              2         100.0


In [22]:

create_classroom("Demo_class2", code="4321", daily_limit=5)


add_student("Demo_class2", "1", "saurav")
add_student("Demo_class2", "2", "lokesh")
add_student("Demo_class2", "3", "dipesh")


mark_attendance("Demo_class2", "1", "4321")
mark_attendance("Demo_class2", "2", "4321")
mark_attendance("Demo_class2", "3", "4321")


attendance_matrix("Demo_class2")


attendance_analytics("Demo_class2")

Class 'Demo_class2' already exists.
Roll 1 already assigned to saurav
Roll 2 already assigned to lokesh
Roll 3 already assigned to dipesh
Incorrect code entered.
Incorrect code entered.
Incorrect code entered.
Top 3 students:
date roll_number    name 2026-03-05 2026-03-06  Present_Count  Attendance %
0              1  saurav          P          P              2         100.0
1              2  lokesh          P          P              2         100.0
2              3  dipesh          P          P              2         100.0

Bottom 3 students:
date roll_number    name 2026-03-05 2026-03-06  Present_Count  Attendance %
0              1  saurav          P          P              2         100.0
1              2  lokesh          P          P              2         100.0
2              3  dipesh          P          P              2         100.0
